# 01 - EDA Metrics

Mục tiêu notebook này là đọc toàn bộ metrics, kiểm tra chất lượng dữ liệu, hiểu schema từng service, và vẽ các metric chính trước khi chạy anomaly detector.

Trong bài này, chúng ta phân tích **metrics trước** vì metrics là tín hiệu định lượng theo thời gian. Metrics giúp trả lời hai câu hỏi đầu:

- **WHEN**: anomaly bắt đầu từ khi nào?
- **WHERE**: service nào có dấu hiệu bất thường trước?

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "g2-data" / "g2").exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing g2-data/g2")

ROOT = find_project_root()
DATA = ROOT / "g2-data" / "g2"
METRICS = DATA / "metrics"
PLOTS = ROOT / "lab" / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

services = {}
for path in sorted(METRICS.glob("*.csv")):
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    services[path.stem] = df

sorted(services)

## 1.1 Kiểm tra schema và số dòng

In [ ]:
overview = []
for service, df in services.items():
    overview.append({
        "service": service,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "start": df["timestamp"].min(),
        "end": df["timestamp"].max(),
        "nulls": int(df.isna().sum().sum()),
        "unique_timestamps": df["timestamp"].nunique(),
    })
pd.DataFrame(overview)

## 1.2 Kiểm tra gap timestamp

In [ ]:
gaps = []
for service, df in services.items():
    diffs = df["timestamp"].diff()
    bad = df[diffs > pd.Timedelta(seconds=30)]
    for idx, row in bad.iterrows():
        prev = df.loc[idx - 1, "timestamp"]
        missing = int((row["timestamp"] - prev).total_seconds() / 30) - 1
        gaps.append({
            "service": service,
            "gap_start_after": prev,
            "next_timestamp": row["timestamp"],
            "missing_30s_points": missing,
        })
pd.DataFrame(gaps)

Kết quả cho thấy tất cả file metrics đều thiếu 60 điểm từ `11:30Z` đến `11:59:30Z`. Gap này cần ghi vào report, nhưng không che mất failure window chính vì incident rõ nhất xảy ra từ sau `14:00Z`.

## 1.3 Thống kê min/median/mean/max từng metric

In [ ]:
rows = []
for service, df in services.items():
    for col in df.columns:
        if col == "timestamp":
            continue
        rows.append({
            "service": service,
            "metric": col,
            "min": df[col].min(),
            "median": df[col].median(),
            "mean": df[col].mean(),
            "max": df[col].max(),
            "max_at": df.loc[df[col].idxmax(), "timestamp"],
        })
metric_summary = pd.DataFrame(rows)
metric_summary

## 1.4 Request rate theo service

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, df in services.items():
    if "http_requests_per_sec" in df:
        ax.plot(df["timestamp"], df["http_requests_per_sec"], label=name, linewidth=1)
ax.set_title("Request rate theo service")
ax.set_ylabel("requests/sec")
ax.grid(True, alpha=0.25)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS / "01_eda_request_rates.png", dpi=160)
plt.show()

**Request rate theo service**

![Request rate theo service](../plots/01_eda_request_rates.png)

## 1.5 Cart-service core metrics

In [ ]:
cart = services["cart-service"].copy()
cart["memory_pct"] = 100 * cart["memory_usage_bytes"] / cart["memory_limit_bytes"]
cart["restart_delta"] = cart["container_restart_count"].diff().fillna(0)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
axes[0].plot(cart["timestamp"], cart["memory_pct"], color="#1f77b4")
axes[0].axhline(70, color="#d62728", linestyle="--", linewidth=1)
axes[0].set_ylabel("memory %")
axes[1].plot(cart["timestamp"], cart["jvm_gc_pause_ms_avg"], color="#9467bd")
axes[1].axhline(100, color="#d62728", linestyle="--", linewidth=1)
axes[1].set_ylabel("GC ms")
axes[2].plot(cart["timestamp"], cart["http_p99_latency_ms"], color="#ff7f0e")
axes[2].set_ylabel("p99 ms")
axes[3].plot(cart["timestamp"], cart["http_5xx_rate"], color="#d62728")
axes[3].bar(cart["timestamp"], cart["restart_delta"] * 5, width=0.015, color="#111111", alpha=0.45)
axes[3].set_ylabel("5xx/restart")
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.suptitle("Cart-service: memory, GC, latency, 5xx và restart")
fig.tight_layout()
fig.savefig(PLOTS / "01_eda_cart_core_metrics.png", dpi=160)
plt.show()

**Cart-service core metrics**

![Cart-service core metrics](../plots/01_eda_cart_core_metrics.png)

Nhìn EDA ban đầu, `cart-service` có chuỗi rất đáng nghi: latency tăng trước, GC tăng, memory tăng, sau đó mới đến restart và 5xx. Đây là dấu hiệu của resource pressure, không chỉ là lỗi HTTP đơn thuần.

## 1.6 Downstream failures

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(services["api-gateway"]["timestamp"], services["api-gateway"]["cart_upstream_error_rate"], label="gateway cart_upstream_error_rate")
ax.plot(services["order-service"]["timestamp"], services["order-service"]["upstream_timeout_rate"], label="order upstream_timeout_rate")
ax.plot(services["payment-service"]["timestamp"], services["payment-service"]["upstream_timeout_rate"], label="payment upstream_timeout_rate")
ax.set_title("Downstream/upstream failures sau khi cart-service bất ổn")
ax.set_ylabel("rate %")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS / "01_eda_downstream_failures.png", dpi=160)
plt.show()

**Downstream failures**

![Downstream failures](../plots/01_eda_downstream_failures.png)

## 1.7 Product-service early suspicious signal

In [ ]:
product = services["product-service"]
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
axes[0].plot(product["timestamp"], product["http_p99_latency_ms"], color="#ff7f0e")
axes[0].axhline(100, color="#d62728", linestyle="--", linewidth=1)
axes[0].set_ylabel("p99 ms")
axes[1].plot(product["timestamp"], product["http_5xx_rate"], color="#d62728")
axes[1].axhline(5, color="#d62728", linestyle="--", linewidth=1)
axes[1].set_ylabel("5xx %")
axes[2].plot(product["timestamp"], product["cpu_usage_percent"], color="#2ca02c")
axes[2].axhline(60, color="#d62728", linestyle="--", linewidth=1)
axes[2].set_ylabel("CPU %")
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.suptitle("Product-service early suspicious signal")
fig.tight_layout()
fig.savefig(PLOTS / "01_eda_product_service_anomaly.png", dpi=160)
plt.show()

**Product-service early suspicious signal**

![Product-service early suspicious signal](../plots/01_eda_product_service_anomaly.png)

`product-service` có anomaly sớm khoảng `03:03Z`. Tuy nhiên ở bước EDA ta chỉ gọi đây là **early suspicious signal / possible trigger**, chưa gọi là root cause vì chưa có log/trace chứng minh quan hệ trực tiếp với cart.